In [1]:
!nvidia-smi

Mon May 18 15:04:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Дообучение Qwen2.5-0.5B для детекции AI-тексто
!pip install -q transformers datasets accelerate peft bitsandbytes scikit-learn pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00


In [4]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    BitsAndBytesConfig, TrainingArguments, Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
import gc

from google.colab import drive
drive.mount('/content/drive')


df = pd.read_csv('/content/drive/MyDrive/LLMTrace_20k_balanced.csv')

Mounted at /content/drive


In [5]:

df['binary_label'] = df['label'].apply(lambda x: 0 if x in [0,3] else 1)
df['is_clean'] = df['label'].isin([0,1])

# Удаляем выбросы
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
def remove_outliers(df, genre_col='style', upper_quantile=0.95):
    cleaned = []
    for genre in df[genre_col].unique():
        human_subset = df[(df[genre_col]==genre) & (df['binary_label']==0)]
        if len(human_subset) > 0:
            thresh = human_subset['word_count'].quantile(upper_quantile)
            human_filtered = human_subset[human_subset['word_count'] <= thresh]
            cleaned.append(human_filtered)
        ai_subset = df[(df[genre_col]==genre) & (df['binary_label']==1)]
        cleaned.append(ai_subset)
    return pd.concat(cleaned, ignore_index=True)

df = remove_outliers(df, upper_quantile=0.95)
df.drop(columns=['word_count'], inplace=True)

print(f"Размер датасета: {len(df)}")
print(f"Распределение меток:\n{df['binary_label'].value_counts()}")
print(f"Жанры: {df['style'].unique().tolist()}")

# 2. РАЗБИЕНИЕ НА TRAIN/TEST
df['stratify'] = df['binary_label'].astype(str) + '_' + df['style']
train_idx, test_idx = train_test_split(
    df.index, test_size=0.2, random_state=42, stratify=df['stratify']
)
train_full = df.loc[train_idx].copy()
test_full = df.loc[test_idx].copy()
train_full.drop(columns=['stratify'], inplace=True)
test_full.drop(columns=['stratify'], inplace=True)

print(f"Train размер: {len(train_full)}")
print(f"Test размер: {len(test_full)}")
print(f"Train метки:\n{train_full['binary_label'].value_counts()}")
print(f"Test метки:\n{test_full['binary_label'].value_counts()}")

test_clean = test_full[test_full['is_clean'] == True]
test_all = test_full

genres = df['style'].unique().tolist()
genre_column = 'style'

print(f"\nТестовые чистые данные: {len(test_clean)}")
print(f"Тестовые все данные: {len(test_all)}")

Размер датасета: 19597
Распределение меток:
binary_label
1    11957
0     7640
Name: count, dtype: int64
Жанры: ['literature', 'scientific', 'publicistic']
Train размер: 15677
Test размер: 3920
Train метки:
binary_label
1    9565
0    6112
Name: count, dtype: int64
Test метки:
binary_label
1    2392
0    1528
Name: count, dtype: int64

Тестовые чистые данные: 3028
Тестовые все данные: 3920


In [10]:

# 3. ПОДГОТОВКА ДАННЫХ ДЛЯ ОБУЧЕНИЯ
model_name = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors=None
    )

def prepare_dataset(df, max_length=512):
    """Преобразует DataFrame в Dataset и токенизирует."""
    dataset = Dataset.from_pandas(df[['text', 'binary_label']])

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding=False,
            max_length=max_length
        )

    dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
    dataset = dataset.rename_column("binary_label", "label")
    dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
    return dataset

train_dataset = prepare_dataset(train_full)
print(f"Train dataset размер: {len(train_dataset)}")

Map:   0%|          | 0/15677 [00:00<?, ? examples/s]

Train dataset размер: 15677


In [11]:

# 4. НАСТРОЙКА МОДЕЛИ
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.pad_token_id = tokenizer.pad_token_id

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForSequenceClassification LOAD REPORT from: Qwen/Qwen2.5-0.5B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 1,083,136 || all params: 495,117,696 || trainable%: 0.2188


In [13]:
# 5. ОБУЧЕНИЕ МОДЕЛИ (ОДИН РАЗ НА ВСЕХ ДАННЫХ)
def compute_metrics(eval_pred):
    """Вычисляет метрики для Trainer."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds, average='binary'),
        'f1_AI': f1_score(labels, preds, pos_label=1),
        'f1_Human': f1_score(labels, preds, pos_label=0),
    }

training_args = TrainingArguments(
    output_dir="./qwen_finetuned",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=50,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("НАЧАЛО ОБУЧЕНИЯ Qwen2.5-0.5B")
print("="*60)

trainer.train()


НАЧАЛО ОБУЧЕНИЯ Qwen2.5-0.5B (1 раз на всех данных)


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,0.559113
100,0.716983
150,0.694872
200,0.548261
250,0.661649
300,0.531813
350,0.470341
400,0.484903
450,0.435815
500,0.451736


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


TrainOutput(global_step=980, training_loss=0.42613680508671975, metrics={'train_runtime': 4548.4402, 'train_samples_per_second': 6.893, 'train_steps_per_second': 0.215, 'total_flos': 3.454294486536192e+16, 'train_loss': 0.42613680508671975, 'epoch': 2.0})

In [2]:

def evaluate_on_dataset(test_df, desc):
    """Тестирует обученную модель на датасете."""
    print(f"\nТестирование на: {desc} (n={len(test_df)})")

    test_dataset = prepare_dataset(test_df)

    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = predictions.label_ids

    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1_AI': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_Human': f1_score(y_true, y_pred, pos_label=0, zero_division=0),
        'precision_AI': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall_AI': recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    }

    print(f"  Accuracy: {metrics['accuracy']:.4f}")
    print(f"  F1 AI: {metrics['f1_AI']:.4f}, F1 Human: {metrics['f1_Human']:.4f}")

    return metrics

# ТЕСТИРОВАНИЕ
results = []

print("1:")
print("="*60)

for genre in genres:
    test_genre = test_clean[test_clean[genre_column] == genre]
    if len(test_genre) > 0:
        metrics = evaluate_on_dataset(
            test_genre,
            f"Чистые данные, жанр: {genre}"
        )
        metrics['case'] = 'Case1_clean_by_genre'
        metrics['genre'] = genre
        metrics['data_type'] = 'clean'
        results.append(metrics)

print("\n" + "="*60)
print("2")

if len(test_clean) > 0:
    metrics = evaluate_on_dataset(
        test_clean,
        "Чистые данные, все жанры"
    )
    metrics['case'] = 'Case2_clean_all_genres'
    metrics['genre'] = 'all'
    metrics['data_type'] = 'clean'
    results.append(metrics)

print("3")
print("="*60)

for genre in genres:
    test_genre = test_all[test_all[genre_column] == genre]
    if len(test_genre) > 0:
        metrics = evaluate_on_dataset(
            test_genre,
            f"Все данные, жанр: {genre}"
        )
        metrics['case'] = 'Case3_all_by_genre'
        metrics['genre'] = genre
        metrics['data_type'] = 'all'
        results.append(metrics)

print("4")
print("="*60)

if len(test_all) > 0:
    metrics = evaluate_on_dataset(
        test_all,
        "Все данные, все жанры"
    )
    metrics['case'] = 'Case4_all_all_genres'
    metrics['genre'] = 'all'
    metrics['data_type'] = 'all'
    results.append(metrics)



1:


NameError: name 'genres' is not defined

In [15]:
df_results = pd.DataFrame(results)
df_results['model'] = 'Qwen2.5-0.5B_Finetuned'

print("РЕЗУЛЬТАТЫ")
print("="*80)

summary_table = df_results[['case', 'data_type', 'genre', 'accuracy', 'f1_AI', 'f1_Human', 'precision_AI', 'recall_AI']]
print(summary_table.to_string(index=False))

df_results.to_csv('/content/drive/MyDrive/qwen_finetuned_results.csv', index=False)

for case in ['Case1_clean_by_genre', 'Case2_clean_all_genres', 'Case3_all_by_genre', 'Case4_all_all_genres']:
    print(f"\n{case}:")
    case_df = df_results[df_results['case'] == case]
    if case == 'Case1_clean_by_genre' or case == 'Case3_all_by_genre':
        for _, row in case_df.iterrows():
            print(f"  {row['genre']}: Acc={row['accuracy']:.4f}, F1_AI={row['f1_AI']:.4f}, F1_Human={row['f1_Human']:.4f}")
        print(f"  СРЕДНЕЕ: Acc={case_df['accuracy'].mean():.4f} ± {case_df['accuracy'].std():.4f}")
    else:
        row = case_df.iloc[0]
        print(f"  Acc={row['accuracy']:.4f}, F1_AI={row['f1_AI']:.4f}, F1_Human={row['f1_Human']:.4f}")


ИТОГОВЫЕ РЕЗУЛЬТАТЫ (Qwen2.5-0.5B после дообучения)
                  case data_type       genre  accuracy    f1_AI  f1_Human  precision_AI  recall_AI
  Case1_clean_by_genre     clean  literature  0.953297 0.947368  0.958025      0.954262   0.940574
  Case1_clean_by_genre     clean  scientific  0.961950 0.968627  0.951662      0.962963   0.974359
  Case1_clean_by_genre     clean publicistic  0.955251 0.960579  0.948258      0.969156   0.952153
Case2_clean_all_genres     clean         all  0.956407 0.959158  0.953258      0.962733   0.955610
    Case3_all_by_genre       all  literature  0.928818 0.936615  0.918834      0.924666   0.948878
    Case3_all_by_genre       all  scientific  0.949087 0.961843  0.923521      0.948864   0.975182
    Case3_all_by_genre       all publicistic  0.920391 0.937773  0.889535      0.926645   0.949171
  Case4_all_all_genres       all         all  0.931122 0.944284  0.909820      0.932355   0.956522

Результаты сохранены в 'qwen_finetuned_results.csv'

ДЕ

In [16]:
model.save_pretrained("/content/drive/MyDrive/qwen_finetuned_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/qwen_finetuned_lora")


Модель сохранена в /content/drive/MyDrive/qwen_finetuned_lora
